# 🖥️ Sigma Coordinate Vertical Grids on Unstructured Grids

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

import parcels
from parcels._datasets.unstructured.generated import sigma_coordinate_lattice_dataset

plt.rcParams["animation.html"] = "jshtml"

## Building the Dataset

For this tutorial, we construct our dataset using the parcels built in {py:func}`parcels._datasets.unstructured.generated.sigma_coordinate_lattice_dataset`, which makes an unstructured horizontally gridded dataset that uses a linear vertical $\sigma$ grid. Here, we assume $\frac{\partial \sigma}{\partial z}$ to be constant, so for any given z coordinate, the equivalent $\sigma$ coordinate is a dimensionless fraction of column height, i.e. $\sigma = \frac{z + \eta}{H +\eta}$, where $H$ is the height of the water column, and $\eta$ is the sea surface height. It is common in $\sigma$ coordinate models to use more advanced vertical grids for which $\frac{\partial \sigma}{\partial z}$ is not constant, this allows the concentration of vertical layers near the surface. Nonetheless, our simple example allows for a vertical coordinate system that adapts to changes in bathymetry and variations in sea surface height. We will demonstrate this shortly but begin by constructing the dataset itself:

In [ ]:
nx = 40  # the number of grid cells in x and y
x_range, y_range = (
    (0.0, 40e3),
    (0.0, 10e3),
)  # we set the bounds of the domain to be a 40km x 10km channel
n_vertical_layers = 10

x_vals = np.linspace(x_range[0], x_range[1], nx)
y_vals = np.linspace(y_range[0], y_range[1], nx)

# we create a time array that goes from hour 0 to hour 12
t = np.arange(13) * 3600.0

# we construct a depth profile that is 10m deep near shore and logarithmically increases to 100m depth offshore
bottom_depth = 10 + 90 * np.log10(1 + 9 * (x_vals / x_range[1]))
bottom_depth = np.broadcast_to(bottom_depth[:, np.newaxis], (nx, nx))

# for illustrative purposes we create 2m surface wave that propogates with time
wavelength, wave_period = 20e3, 6 * 3600.0
amplitude = 2
wave = amplitude * np.sin(
    2.0
    * np.pi
    * (
        np.broadcast_to(x_vals[:, np.newaxis], (nx, nx)) / wavelength
        - t[:, np.newaxis, np.newaxis] / wave_period
    )
)

# construct the dataset itself
ds = sigma_coordinate_lattice_dataset(
    nx, x_range, y_range, n_vertical_layers + 1, bottom_depth, eta=wave
)
ds

We can now compare the evolution of the vertical grid in $\sigma$ space to the vertical grid in z space. In z space, the vertical grid spacing is much smaller near shore, where the domain is shallower, this demonstrates the strength of a terrain following vertical grid. We can also see that the vertical has time dependency in z space.

In [ ]:
time = ds["time"].values.astype("datetime64[s]")
z_moving = ds["zf"].values  # interface depths, shape (time, zf, n_node)
sigma = np.linspace(0.0, 1.0, n_vertical_layers + 1)

node_x, node_y = ds.uxgrid.node_lon.values, ds.uxgrid.node_lat.values
centre_line = np.isclose(
    node_y, y_vals[nx // 2]
)  # the row of nodes closest to the middle of the channel
xs = node_x[centre_line] / 1e3

fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(xs, np.broadcast_to(sigma, (xs.size, sigma.size)), "k", linewidth=0.8)
ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Layer interfaces in $\sigma$ space")

z_lines = ax_z.plot(xs, z_moving[0][:, centre_line].T, "k", linewidth=0.8)
ax_z.set_ylabel("depth [m]")
z_title = ax_z.set_title(f"Layer interfaces in z space at {time[0]}")

for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()


def animate(snapshot):
    for line, interface_depths in zip(z_lines, z_moving[snapshot][:, centre_line]):
        line.set_ydata(interface_depths)
    z_title.set_text(f"Layer interfaces in z space at {time[snapshot]}")


anim = FuncAnimation(fig, animate, frames=time.size, interval=500)
plt.close()
anim

## Building the dataset and `FieldSet`

We prescribe a uniform along channel velocity of $u_0 = 0.5$ m/s, with $v = w = 0$. All three components sit on the mesh nodes at the layer interfaces (`zf`), so Parcels interpolates them barycentrically in the horizontal and linearly in the vertical.

To advect the particles on sigma grid, for which the z grid varies in time, we use a 3D `zf` coordinate with dimensions of `(time, zf, n_node)`. Parcels then uses it to find each particle's own column at the current time, before searching that column for the particle's layer. `parcels.FieldSet.from_ugrid_conventions()` also expects a `zc` dimension for the layer centres, so we add the layer-centre depths as well. We pass `mesh="flat"` because the coordinates are Cartesian meters.

In [ ]:
u0 = 0.5
ds["U"].values[:] = u0
ds

In [ ]:
fieldset = parcels.FieldSet.from_ugrid_conventions(ds, mesh="flat")
fieldset.describe()

## Release the particles

We release 10 particles at the first snapshot near the shallow end of the channel, at $x = 2$ km on the centre line, at depths from 3 m to 22 m. The water column there is about 22.5 m deep, and the shallowest particle starts below the deepest wave trough (2 m), so every particle will remain in the water column throughout the simulation.

In [ ]:
n_particles = 10
x0 = np.full(n_particles, 2e3)
y0 = np.full(n_particles, 5e3)
z0 = np.linspace(3.0, 22.0, n_particles)  # depths in m

pset = parcels.ParticleSet(
    fieldset=fieldset,
    pclass=parcels.Particle,
    t=np.full(n_particles, time[0]),
    z=z0,
    y=y0,
    x=x0,
)

In [ ]:
H = z_moving[0, -1]  # the bottom interface sits at the bottom depth, shape (n_node,)
eta = -z_moving[:, 0]  # the surface interface sits at depth -eta, shape (time, n_node)
x_line = node_x[centre_line]
H0 = np.interp(x0, x_line, H[centre_line])
eta0 = np.interp(x0, x_line, eta[0, centre_line])
sigma0 = (z0 + eta0) / (H0 + eta0)  # initial sigma coordinate of each particle

colors = [f"C{p}" for p in range(n_particles)]

fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(xs, np.broadcast_to(sigma, (xs.size, sigma.size)), "k", linewidth=0.8)
ax_sigma.scatter(x0 / 1e3, sigma0, color=colors, zorder=3)
ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Initial particle positions in $\sigma$ space")

ax_z.plot(xs, z_moving[0][:, centre_line].T, "k", linewidth=0.8)
ax_z.scatter(x0 / 1e3, z0, color=colors, zorder=3)
ax_z.set_ylabel("depth [m]")
ax_z.set_title(f"Initial particle positions in z space at {time[0]}")

for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()
plt.show()

## Advect the particles

We advect the particles for 12 hours using the 3D fourth-order Runge-Kutta kernel `parcels.kernels.AdvectionRK4_3D` with a 5-minute time step, and write their positions every hour. 

In [ ]:
outputfile = parcels.ParticleFile(
    path="sigma_particles.parquet", outputdt=np.timedelta64(1, "h"), mode="w"
)

pset.execute(
    [parcels.kernels.AdvectionRK4_3D],
    runtime=np.timedelta64(12, "h"),
    dt=np.timedelta64(5, "m"),
    output_file=outputfile,
)

In [ ]:
df = parcels.read_particlefile("sigma_particles.parquet").sort(["t", "particle_id"])
output_times = df["t"].unique().sort()
n_frames = len(output_times)
px = df["x"].to_numpy().reshape(n_frames, n_particles)
pz = df["z"].to_numpy().reshape(n_frames, n_particles)

# calcualte the sigma coordinate of every particle at every output time
snapshots = np.searchsorted(time, output_times.to_numpy(), side="right") - 1
psigma = np.empty_like(pz)
for frame, snapshot in enumerate(snapshots):
    eta_p = np.interp(px[frame], x_line, eta[snapshot, centre_line])
    psigma[frame] = (pz[frame] + eta_p) / (
        np.interp(px[frame], x_line, H[centre_line]) + eta_p
    )

colors = [f"C{p}" for p in range(n_particles)]

fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(xs, np.broadcast_to(sigma, (xs.size, sigma.size)), "k", linewidth=0.8)
z_lines = ax_z.plot(xs, z_moving[0][:, centre_line].T, "k", linewidth=0.8)
sigma_trails = [ax_sigma.plot([], [], color=c, linewidth=1.5)[0] for c in colors]
z_trails = [ax_z.plot([], [], color=c, linewidth=1.5)[0] for c in colors]
sigma_dots = ax_sigma.scatter(px[0] / 1e3, psigma[0], c=colors, zorder=3)
z_dots = ax_z.scatter(px[0] / 1e3, pz[0], c=colors, zorder=3)

ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Particles in $\sigma$ space")
ax_z.set_ylabel("depth [m]")
z_title = ax_z.set_title("")
for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()


def animate(frame):
    for p in range(n_particles):
        sigma_trails[p].set_data(px[: frame + 1, p] / 1e3, psigma[: frame + 1, p])
        z_trails[p].set_data(px[: frame + 1, p] / 1e3, pz[: frame + 1, p])
    sigma_dots.set_offsets(np.c_[px[frame] / 1e3, psigma[frame]])
    z_dots.set_offsets(np.c_[px[frame] / 1e3, pz[frame]])
    for line, interface_depths in zip(
        z_lines, z_moving[snapshots[frame]][:, centre_line]
    ):
        line.set_ydata(interface_depths)
    z_title.set_text(f"Particles in z space at {output_times[frame]}")


animate(0)
anim = FuncAnimation(fig, animate, frames=n_frames, interval=300)
plt.close()
anim

We can see that, as expected, the particles move horizontally in x and cross the terrain following depth levels. However, in z space, they maintain their exact height as expected.